In [1]:
# Script to read the CPM file with the EnsembleIDs to transform them to gene names for the deconvolution.

import pandas as pd

# Load mapping file - specify header row and that first column is index
mapping = pd.read_csv('20250405_Both_LessTissueV2_2Median-Unique.tsv', sep='\t', index_col=0)

print("Mapping index (should be Ensembl IDs):", mapping.index[:5])

# The first column after index should be gene names
print("Gene names (first column):", mapping.columns[0])
print(mapping.iloc[:5, 0])  # first 5 gene names

# Now create the 'geneID_no_version' by stripping the index (Ensembl IDs)
mapping.index = mapping.index.str.strip()  # strip whitespace if any
mapping['geneID_no_version'] = mapping.index.str.replace(r'\.\d+$', '', regex=True)

print("Ensembl IDs stripped:", mapping['geneID_no_version'][:5])

# Now, for your counts file, load it:
counts = pd.read_csv('GSE225221_cfrna_counts_CPM.txt', sep='\t')
counts['geneID_no_version'] = counts['geneID'].str.replace(r'\.\d+$', '', regex=True)

# Map Ensembl IDs (no version) to gene names
gene_map = mapping.iloc[:, 0].copy()  # first column of mapping: gene names
gene_map.index = mapping['geneID_no_version']  # use stripped Ensembl IDs as index

# Map counts to gene names
counts['gene_name'] = counts['geneID_no_version'].map(gene_map)

# Count unmapped
unmapped = counts['gene_name'].isna().sum()
print(f"Number of unmapped transcripts: {unmapped}")

# Optional: check some unmapped geneIDs
print("Example unmapped geneIDs:", counts.loc[counts['gene_name'].isna(), 'geneID_no_version'].unique()[:20])

counts['geneID_no_version'] = counts['geneID_no_version'].str.strip()
mapping['geneID_no_version'] = mapping['geneID_no_version'].str.strip()
mapping.index = mapping.index.str.strip()


Mapping index (should be Ensembl IDs): Index(['ENSG00000223972.5', 'ENSG00000227232.5', 'ENSG00000278267.1',
       'ENSG00000243485.5', 'ENSG00000237613.2'],
      dtype='object')
Gene names (first column): GeneID
ENSG00000223972.5        DDX11L1
ENSG00000227232.5         WASH7P
ENSG00000278267.1      MIR6859-1
ENSG00000243485.5    MIR1302-2HG
ENSG00000237613.2        FAM138A
Name: GeneID, dtype: object
Ensembl IDs stripped: ENSG00000223972.5    ENSG00000223972
ENSG00000227232.5    ENSG00000227232
ENSG00000278267.1    ENSG00000278267
ENSG00000243485.5    ENSG00000243485
ENSG00000237613.2    ENSG00000237613
Name: geneID_no_version, dtype: object
Number of unmapped transcripts: 6688
Example unmapped geneIDs: ['ENSG00000284332' 'ENSG00000239945' 'ENSG00000286448' 'ENSG00000284733'
 'ENSG00000278791' 'ENSG00000284662' 'ENSG00000285268' 'ENSG00000288531'
 'ENSG00000242590' 'ENSG00000285812' 'ENSG00000264293' 'ENSG00000284740'
 'ENSG00000286989' 'ENSG00000268575' 'ENSG00000287356' 'ENSG0000

Matching the EnsemblID with the Gene names using the GTEx file results in 6k unmapped genes which is a bit too many. Re-naming using the gencode gtf file leads to match less unmapped genes (327), which are dropped. Moreover, 1459 gene names are duplicates so these transcripts are sum up to de-duplicate and allow deconvolution.

In [3]:
import pandas as pd
import re

# Step 1: Parse the GTF file for gene_id and gene_name
def parse_gtf(gtf_file):
    gene_map = {}
    with open(gtf_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue  # skip headers/comments
            fields = line.strip().split('\t')
            if fields[2] == 'gene':  # only parse gene entries
                attr_field = fields[8]
                # Extract gene_id and gene_name from the attributes column
                gene_id_match = re.search(r'gene_id "([^"]+)"', attr_field)
                gene_name_match = re.search(r'gene_name "([^"]+)"', attr_field)
                if gene_id_match and gene_name_match:
                    gene_id = gene_id_match.group(1)
                    gene_name = gene_name_match.group(1)
                    # Strip version number from gene_id (e.g., ENSG00000290825.1 -> ENSG00000290825)
                    gene_id_no_version = gene_id.split('.')[0]
                    gene_map[gene_id_no_version] = gene_name
    return gene_map

# Load your GTF and create mapping dictionary
gtf_file = 'Raw-Published/gencode.v46.annotation.gtf'
gene_map = parse_gtf(gtf_file)

print(f"Total genes in mapping: {len(gene_map)}")

# Step 2: Load counts matrix and strip version from gene IDs
counts = pd.read_csv('GSE225221_cfrna_counts_CPM.txt', sep='\t')

# Assuming your counts matrix has a column 'geneID' with Ensembl IDs with versions
counts['geneID_no_version'] = counts['geneID'].str.split('.').str[0]

# Step 3: Map gene names from GTF dictionary
counts['gene_name'] = counts['geneID_no_version'].map(gene_map)

# Step 4: Check how many genes remain unmapped
num_unmapped = counts['gene_name'].isna().sum()
print(f"Unmapped genes after GTF mapping: {num_unmapped}")

# Step 5: Check for duplicate gene names
num_duplicates = counts.duplicated(subset='gene_name').sum()
print(f"Number of duplicate gene names: {num_duplicates}")

# Step 6: Set gene_name as index and drop unnecessary columns
counts.set_index('gene_name', inplace=True)
counts.drop(['geneID', 'geneID_no_version'], axis=1, inplace=True)

# Step 7: Sum duplicate gene names
counts = counts.groupby(counts.index).sum()

# Save the renamed counts file
counts.to_csv('GSE225221_cfrna_counts_CPM_GeneNames.txt', sep='\t')

print("Gene renaming and deduplication complete. File saved.")


Total genes in mapping: 63086
Unmapped genes after GTF mapping: 343
Number of duplicate gene names: 1585
Gene renaming and deduplication complete. File saved.
